In [134]:
import pandas as pd
import numpy as np

In [135]:
df = pd.read_csv('../data/processed/llm_telemetry_v1_cleaned_2026-05-26.csv')
df.head()

,timestamp,session_id,model_name,input_tokens,output_tokens,latency_ms,status_code,gpu_cluster
0,2026-05-21 06:24:00,USR_1409,Gpt-3.5-Turbo,1136,440,1492.41,200,gcp-asia
1,2026-05-26 04:55:00,USR_2424,Gemini_1.5,340,463,1211.95,429,Unknown
2,2026-05-26 20:23:00,USR_1434,Gemini-1.5-Pro,1133,876,2389.46,200,gcp-asia
3,2026-05-25 02:39:00,USR_5557,Gpt-4,469,318,944.24,200,gcp-asia
4,2026-05-23 19:54:00,USR_2674,Gpt-4,757,861,2241.32,200,aws-us-west


In [136]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1030 entries, 0 to 1029
Data columns (total 8 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   timestamp      1030 non-null   str    
 1   session_id     1030 non-null   str    
 2   model_name     1030 non-null   str    
 3   input_tokens   1030 non-null   int64  
 4   output_tokens  1030 non-null   int64  
 5   latency_ms     1030 non-null   float64
 6   status_code    1030 non-null   int64  
 7   gpu_cluster    1030 non-null   str    
dtypes: float64(1), int64(3), str(4)
memory usage: 64.5 KB


In [137]:
df.head()

,timestamp,session_id,model_name,input_tokens,output_tokens,latency_ms,status_code,gpu_cluster
0,2026-05-21 06:24:00,USR_1409,Gpt-3.5-Turbo,1136,440,1492.41,200,gcp-asia
1,2026-05-26 04:55:00,USR_2424,Gemini_1.5,340,463,1211.95,429,Unknown
2,2026-05-26 20:23:00,USR_1434,Gemini-1.5-Pro,1133,876,2389.46,200,gcp-asia
3,2026-05-25 02:39:00,USR_5557,Gpt-4,469,318,944.24,200,gcp-asia
4,2026-05-23 19:54:00,USR_2674,Gpt-4,757,861,2241.32,200,aws-us-west


In [138]:
# Extract hour from timestamp to analyze workload patterns across different times of the day.

df['hour_of_day'] = pd.to_datetime(df['timestamp']).dt.hour
df.head()

,timestamp,session_id,model_name,input_tokens,output_tokens,latency_ms,status_code,gpu_cluster,hour_of_day
0,2026-05-21 06:24:00,USR_1409,Gpt-3.5-Turbo,1136,440,1492.41,200,gcp-asia,6
1,2026-05-26 04:55:00,USR_2424,Gemini_1.5,340,463,1211.95,429,Unknown,4
2,2026-05-26 20:23:00,USR_1434,Gemini-1.5-Pro,1133,876,2389.46,200,gcp-asia,20
3,2026-05-25 02:39:00,USR_5557,Gpt-4,469,318,944.24,200,gcp-asia,2
4,2026-05-23 19:54:00,USR_2674,Gpt-4,757,861,2241.32,200,aws-us-west,19


In [139]:
df['model_name'].unique()

<StringArray>
[ 'Gpt-3.5-Turbo',     'Gemini_1.5', 'Gemini-1.5-Pro',          'Gpt-4',
  'Claude-3-Opus',           'Gpt4',    'Gpt-4-Turbo',       'Claude-3']
Length: 8, dtype: str

In [140]:
# Encode categorical model names into numerical binary flags to enable model-specific training.

dummies_model = pd.get_dummies(df['model_name'], prefix= 'model', dtype= int)
df = df.join(dummies_model)
df.head()

,timestamp,session_id,model_name,input_tokens,output_tokens,latency_ms,status_code,gpu_cluster,hour_of_day,model_Claude-3,model_Claude-3-Opus,model_Gemini-1.5-Pro,model_Gemini_1.5,model_Gpt-3.5-Turbo,model_Gpt-4,model_Gpt-4-Turbo,model_Gpt4
0,2026-05-21 06:24:00,USR_1409,Gpt-3.5-Turbo,1136,440,1492.41,200,gcp-asia,6,0,0,0,0,1,0,0,0
1,2026-05-26 04:55:00,USR_2424,Gemini_1.5,340,463,1211.95,429,Unknown,4,0,0,0,1,0,0,0,0
2,2026-05-26 20:23:00,USR_1434,Gemini-1.5-Pro,1133,876,2389.46,200,gcp-asia,20,0,0,1,0,0,0,0,0
3,2026-05-25 02:39:00,USR_5557,Gpt-4,469,318,944.24,200,gcp-asia,2,0,0,0,0,0,1,0,0
4,2026-05-23 19:54:00,USR_2674,Gpt-4,757,861,2241.32,200,aws-us-west,19,0,0,0,0,0,1,0,0


In [141]:
# Create a binary error flag based on non-200 status codes for failure analysis.

df['is_error'] = np.where(df['status_code'] == 200, 0, 1)
df.head()

,timestamp,session_id,model_name,input_tokens,output_tokens,latency_ms,status_code,gpu_cluster,hour_of_day,model_Claude-3,model_Claude-3-Opus,model_Gemini-1.5-Pro,model_Gemini_1.5,model_Gpt-3.5-Turbo,model_Gpt-4,model_Gpt-4-Turbo,model_Gpt4,is_error
0,2026-05-21 06:24:00,USR_1409,Gpt-3.5-Turbo,1136,440,1492.41,200,gcp-asia,6,0,0,0,0,1,0,0,0,0
1,2026-05-26 04:55:00,USR_2424,Gemini_1.5,340,463,1211.95,429,Unknown,4,0,0,0,1,0,0,0,0,1
2,2026-05-26 20:23:00,USR_1434,Gemini-1.5-Pro,1133,876,2389.46,200,gcp-asia,20,0,0,1,0,0,0,0,0,0
3,2026-05-25 02:39:00,USR_5557,Gpt-4,469,318,944.24,200,gcp-asia,2,0,0,0,0,0,1,0,0,0
4,2026-05-23 19:54:00,USR_2674,Gpt-4,757,861,2241.32,200,aws-us-west,19,0,0,0,0,0,1,0,0,0


In [142]:
df['gpu_cluster'].unique()

<StringArray>
['gcp-asia', 'Unknown', 'aws-us-west', 'aws-us-east', 'azure-eu']
Length: 5, dtype: str

In [143]:
# Transform cluster location data into numerical features for infrastructure performance modeling.

dummies_gpu_cluster = pd.get_dummies(df['gpu_cluster'], prefix= "gpu", dtype= int)
df = df.join(dummies_gpu_cluster)
df.head()

,timestamp,session_id,model_name,input_tokens,output_tokens,latency_ms,status_code,gpu_cluster,hour_of_day,model_Claude-3,...,model_Gpt-3.5-Turbo,model_Gpt-4,model_Gpt-4-Turbo,model_Gpt4,is_error,gpu_Unknown,gpu_aws-us-east,gpu_aws-us-west,gpu_azure-eu,gpu_gcp-asia
0,2026-05-21 06:24:00,USR_1409,Gpt-3.5-Turbo,1136,440,1492.41,200,gcp-asia,6,0,...,1,0,0,0,0,0,0,0,0,1
1,2026-05-26 04:55:00,USR_2424,Gemini_1.5,340,463,1211.95,429,Unknown,4,0,...,0,0,0,0,1,1,0,0,0,0
2,2026-05-26 20:23:00,USR_1434,Gemini-1.5-Pro,1133,876,2389.46,200,gcp-asia,20,0,...,0,0,0,0,0,0,0,0,0,1
3,2026-05-25 02:39:00,USR_5557,Gpt-4,469,318,944.24,200,gcp-asia,2,0,...,0,1,0,0,0,0,0,0,0,1
4,2026-05-23 19:54:00,USR_2674,Gpt-4,757,861,2241.32,200,aws-us-west,19,0,...,0,1,0,0,0,0,0,1,0,0


In [144]:
# Calculate total inference cost using weighted token consumption rates for financial monitoring.

df['total_cost_used'] = (df['input_tokens'] * 0.00001) + (df['output_tokens'] * 0.00003)
df.head()

,timestamp,session_id,model_name,input_tokens,output_tokens,latency_ms,status_code,gpu_cluster,hour_of_day,model_Claude-3,...,model_Gpt-4,model_Gpt-4-Turbo,model_Gpt4,is_error,gpu_Unknown,gpu_aws-us-east,gpu_aws-us-west,gpu_azure-eu,gpu_gcp-asia,total_cost_used
0,2026-05-21 06:24:00,USR_1409,Gpt-3.5-Turbo,1136,440,1492.41,200,gcp-asia,6,0,...,0,0,0,0,0,0,0,0,1,0.02456
1,2026-05-26 04:55:00,USR_2424,Gemini_1.5,340,463,1211.95,429,Unknown,4,0,...,0,0,0,1,1,0,0,0,0,0.01729
2,2026-05-26 20:23:00,USR_1434,Gemini-1.5-Pro,1133,876,2389.46,200,gcp-asia,20,0,...,0,0,0,0,0,0,0,0,1,0.03761
3,2026-05-25 02:39:00,USR_5557,Gpt-4,469,318,944.24,200,gcp-asia,2,0,...,1,0,0,0,0,0,0,0,1,0.01423
4,2026-05-23 19:54:00,USR_2674,Gpt-4,757,861,2241.32,200,aws-us-west,19,0,...,1,0,0,0,0,0,1,0,0,0.03340


In [145]:
# Remove identifier columns to streamline the final dataset for machine learning input.

df.drop(['timestamp', 'session_id'], axis= 1, inplace= True)
df.head()

,model_name,input_tokens,output_tokens,latency_ms,status_code,gpu_cluster,hour_of_day,model_Claude-3,model_Claude-3-Opus,model_Gemini-1.5-Pro,...,model_Gpt-4,model_Gpt-4-Turbo,model_Gpt4,is_error,gpu_Unknown,gpu_aws-us-east,gpu_aws-us-west,gpu_azure-eu,gpu_gcp-asia,total_cost_used
0,Gpt-3.5-Turbo,1136,440,1492.41,200,gcp-asia,6,0,0,0,...,0,0,0,0,0,0,0,0,1,0.02456
1,Gemini_1.5,340,463,1211.95,429,Unknown,4,0,0,0,...,0,0,0,1,1,0,0,0,0,0.01729
2,Gemini-1.5-Pro,1133,876,2389.46,200,gcp-asia,20,0,0,1,...,0,0,0,0,0,0,0,0,1,0.03761
3,Gpt-4,469,318,944.24,200,gcp-asia,2,0,0,0,...,1,0,0,0,0,0,0,0,1,0.01423
4,Gpt-4,757,861,2241.32,200,aws-us-west,19,0,0,0,...,1,0,0,0,0,0,1,0,0,0.03340


In [146]:
# Export the final feature-engineered dataset for subsequent model training.

df.to_csv('../data/processed/llm_telemetry_v1_features_added_2026-05-26.csv', index= False)